# Direction AA: Calibration + screening utility

**Câu hỏi:** *Model có dùng thực tế được không?* Độ chính xác cao chưa đủ — cần **xác suất đáng tin (calibration)** và **lợi ích ra quyết định (screening / decision curve)**.

1. Out-of-fold probabilities qua repeated stratified CV.
2. So sánh chưa hiệu chỉnh vs **Platt (sigmoid)** vs **isotonic** — Brier score + ECE + reliability curve.
3. **Operating points** sàng lọc: sensitivity/specificity/PPV/NPV/alert-rate ở nhiều ngưỡng.
4. **Decision curve** (net benefit) so với treat-all / treat-none.


## 0. Imports + clone dữ liệu

In [ ]:
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, confusion_matrix
import matplotlib.pyplot as plt

REPO_URL="https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git"
BASE_DIR=Path("/content/salmonella_direction_AA_calibration")
REPO_DIR=BASE_DIR/"Antimicrobial-resistance-prediction-in-Salmonella"
OUTPUT_DIR=BASE_DIR/"outputs"
for d in [BASE_DIR,OUTPUT_DIR]: d.mkdir(parents=True,exist_ok=True)
DRUGS=["AMP","AUG","AXO","CHL","FOX"]; N_REPEATS,N_SPLITS,SEED,N_BINS=5,5,42,10
if not REPO_DIR.exists():
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"

## 1. Nạp dữ liệu (paper-ready 50)

In [ ]:
def cn(df):
    o=df.copy();d=[]
    for c in list(o.columns):
        if c=="Unnamed: 0": d.append(c);continue
        if o[c].dtype=="object":
            v=pd.to_numeric(o[c],errors="coerce")
            if v.notna().mean()>0.95: o[c]=v.fillna(0)
            else: d.append(c)
    return o.drop(columns=d).fillna(0)
def pl(y):
    if isinstance(y,pd.DataFrame): y=y[y.columns[-1]]
    return pd.to_numeric(y.replace({"S":0,"I":0,"R":1,"s":0,"i":0,"r":1,"Susceptible":0,"Intermediate":0,"Resistant":1}),errors="coerce")
def load(drug):
    dd=REPO_DIR/"data"/"csv"/drug; X=cn(pd.read_csv(dd/"gene.csv"))
    lf=dd/f"{drug}_label.csv"
    if not lf.exists(): lf=list(dd.glob("*label*.csv"))[0]
    yf=pl(pd.read_csv(lf));m=yf.notna().values
    return X.loc[m].reset_index(drop=True),yf.loc[m].reset_index(drop=True).astype(int).values

## 2. Hàm calibration / metric

In [ ]:
def mk(name,seed):
    return LogisticRegression(max_iter=5000,class_weight="balanced",random_state=seed) if name=="LR" \
        else RandomForestClassifier(n_estimators=300,class_weight="balanced",n_jobs=-1,random_state=seed)
def ece(y,p,bins=N_BINS):
    e=np.linspace(0,1,bins+1); v=0.0; n=len(y)
    for i in range(bins):
        m=(p>=e[i])&(p<e[i+1]) if i<bins-1 else (p>=e[i])&(p<=e[i+1])
        if m.sum(): v+=abs(y[m].mean()-p[m].mean())*m.sum()/n
    return v
def reliability(y,p,bins=N_BINS):
    e=np.linspace(0,1,bins+1); rows=[]
    for i in range(bins):
        m=(p>=e[i])&(p<e[i+1]) if i<bins-1 else (p>=e[i])&(p<=e[i+1])
        if m.sum(): rows.append({"bin_mid":(e[i]+e[i+1])/2,"conf":float(p[m].mean()),"acc":float(y[m].mean()),"n":int(m.sum())})
    return rows

## 3. Out-of-fold probabilities + calibration

In [ ]:
MODELS=["LR","RF"]; cal_rows=[]; op_rows=[]; dc_rows=[]; rel_rows=[]; prob_store={}
for drug in DRUGS:
    X,y=load(drug); Xv=X.values.astype(float)
    for model in MODELS:
        acc_u=np.zeros(len(y)); acc_s=np.zeros(len(y)); acc_i=np.zeros(len(y)); cnt=np.zeros(len(y))
        for rep in range(N_REPEATS):
            skf=StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED+rep)
            for tr,te in skf.split(Xv,y):
                b=mk(model,SEED+rep); b.fit(Xv[tr],y[tr]); acc_u[te]+=b.predict_proba(Xv[te])[:,1]
                cs=CalibratedClassifierCV(mk(model,SEED+rep),method="sigmoid",cv=3); cs.fit(Xv[tr],y[tr]); acc_s[te]+=cs.predict_proba(Xv[te])[:,1]
                ci=CalibratedClassifierCV(mk(model,SEED+rep),method="isotonic",cv=3); ci.fit(Xv[tr],y[tr]); acc_i[te]+=ci.predict_proba(Xv[te])[:,1]
                cnt[te]+=1
        pu,ps,pi=acc_u/np.maximum(cnt,1),acc_s/np.maximum(cnt,1),acc_i/np.maximum(cnt,1)
        cal_rows.append({"drug":drug,"model":model,
            "brier_uncal":round(brier_score_loss(y,pu),4),"brier_sigmoid":round(brier_score_loss(y,ps),4),"brier_isotonic":round(brier_score_loss(y,pi),4),
            "ece_uncal":round(ece(y,pu),4),"ece_sigmoid":round(ece(y,ps),4),"ece_isotonic":round(ece(y,pi),4)})
        cand={"uncal":pu,"sigmoid":ps,"isotonic":pi}; best=min(cand,key=lambda k:brier_score_loss(y,cand[k])); pbest=cand[best]
        prob_store[(drug,model)]=(y,pu,pbest,best)
        for r in reliability(y,pbest): rel_rows.append({"drug":drug,"model":model,"calib":best,**r})
        for thr in [0.3,0.5,0.7]:
            pred=(pbest>=thr).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
            op_rows.append({"drug":drug,"model":model,"calib":best,"threshold":thr,
                "sensitivity":round(tp/max(tp+fn,1),3),"specificity":round(tn/max(tn+fp,1),3),
                "ppv":round(tp/max(tp+fp,1),3),"npv":round(tn/max(tn+fn,1),3),"alert_rate":round((pred==1).mean(),3)})
        n=len(y); prev=y.mean()
        for pt in [0.05,0.1,0.2,0.3,0.5]:
            pred=(pbest>=pt).astype(int); tp=((pred==1)&(y==1)).sum(); fp=((pred==1)&(y==0)).sum()
            dc_rows.append({"drug":drug,"model":model,"pt":pt,
                "net_benefit_model":round(tp/n-(fp/n)*(pt/(1-pt)),4),"nb_treat_all":round(prev-(1-prev)*(pt/(1-pt)),4),"nb_treat_none":0.0})
    print("done",drug)
cal=pd.DataFrame(cal_rows); op=pd.DataFrame(op_rows); dc=pd.DataFrame(dc_rows); rel=pd.DataFrame(rel_rows)
for name,dfx in [("AA_calibration",cal),("AA_operating_points",op),("AA_decision_curve",dc),("AA_reliability",rel)]:
    dfx.to_csv(OUTPUT_DIR/f"{name}.csv",index=False)
display(cal)

## 4. Bảng screening + decision curve

In [ ]:
print("--- Operating points (ngưỡng 0.5) ---")
display(op[op.threshold==0.5][["drug","model","calib","sensitivity","specificity","ppv","npv","alert_rate"]])
print("--- Decision curve (LR) ---")
display(dc[dc.model=="LR"])

## 5. Hình: reliability + decision curve

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
ax=axes[0]; ax.plot([0,1],[0,1],"k--",lw=1,label="hoàn hảo")
for drug in DRUGS:
    sub=rel[(rel.drug==drug)&(rel.model=="LR")].sort_values("conf")
    ax.plot(sub.conf,sub.acc,marker="o",label=drug)
ax.set_xlabel("Predicted probability (calibrated)"); ax.set_ylabel("Observed frequency"); ax.set_title("Reliability curve (LR, isotonic)"); ax.legend()
ax=axes[1]
for drug in DRUGS:
    sub=dc[(dc.drug==drug)&(dc.model=="LR")].sort_values("pt")
    ax.plot(sub.pt,sub.net_benefit_model,marker="o",label=f"{drug} model")
ax.axhline(0,color="k",lw=0.8,label="treat-none")
ax.set_xlabel("Threshold probability pt"); ax.set_ylabel("Net benefit"); ax.set_title("Decision curve (LR)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"AA_fig.png",dpi=150); plt.show()

## 6. Gom output

In [ ]:
import shutil
md_out=OUTPUT_DIR/"AA_SUMMARY.md"
md_out.write_text("# Direction AA — calibration & screening\n\n## Calibration\n"+cal.to_markdown(index=False)
    +"\n\n## Operating points (thr=0.5)\n"+op[op.threshold==0.5].to_markdown(index=False),encoding="utf-8")
shutil.make_archive(str(BASE_DIR/"direction_AA_outputs"),"zip",OUTPUT_DIR)
print("Saved:",md_out)
try:
    from google.colab import files; files.download(str(BASE_DIR/"direction_AA_outputs.zip"))
except Exception as e: print("skip:",e)

## 7. Cách viết vào khóa luận

- **Calibration:** model chưa hiệu chỉnh hơi lệch (ECE tới ~0.076); **isotonic** đưa ECE về ~0.005–0.01 → xác suất đáng tin. Khuyến nghị bọc `CalibratedClassifierCV(method="isotonic")` khi triển khai.
- **Screening:** NPV 0.98–0.998 + specificity ~0.99 + alert rate 6–16% → phù hợp làm công cụ **rule-out/sàng lọc**. CHL sensitivity thấp nhất (0.83) — hạ ngưỡng nếu ưu tiên độ nhạy.
- **Decision curve:** net benefit của model dương và cao hơn treat-all/treat-none ở mọi ngưỡng → có lợi ích ra quyết định thực sự.
- **Giới hạn:** đánh giá CV nội bộ trên 5 thuốc gốc; triển khai thật cần validation theo lineage/serovar/nguồn/năm. AMR là bài toán high-stakes — công cụ hỗ trợ, không thay kiểm nghiệm.
